<a href="https://colab.research.google.com/github/ryaposov-nik/hse_python_project/blob/main/%D0%9F%D1%80%D0%BE%D0%B5%D0%BA%D1%82_%D0%BF%D0%BE_superjob.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Импорт модулей

In [ ]:
import json
import re
import requests as rq
import time
import pandas as pd
from requests.exceptions import ChunkedEncodingError, ConnectionError, Timeout



---


# Получение данных

Документация API SuperJob: https://api.superjob.ru/

## Функции для получения основных данных

In [ ]:
def auth():
  code = rq.post('https://api.superjob.ru/2.0/oauth2/password/',
      {'login': 'ryaposov.nik@mail.ru',
      'password': 'paroldlyaproekta',
      'client_id': 4155,
      'client_secret': 'v3.r.139623701.59077548bf42b0eca83b5250419eee576949068c.24f8e3becdb3699a0c88d5b3f606f4a0e43278df'})
  return code

def get_vacancy(page, time_start, time_finish):
  for attempt in range(5):
    try:
      result = rq.get(
      'https://api.superjob.ru/2.0/vacancies/',
      params={
          #'period': 0,
          'no_agreement': 1,
          'date_published_from': time_start,
          'date_published_to': time_finish,
          'order_field': 'date',
          'order_direction': 'desc',
          'page': page
          },
      headers={
          'Authorization': f'{code.json()['token_type']} {code.json()['access_token']}',
          'X-Api-App-Id': 'v3.r.139623701.59077548bf42b0eca83b5250419eee576949068c.24f8e3becdb3699a0c88d5b3f606f4a0e43278df'
          })
      return result
    except (ChunkedEncodingError, ConnectionError, Timeout):
      print(f'Случилась ошибка при получении данных, повторяем попытку. Попытка {attempt+1}')
      time.sleep(1 + attempt)

### Получаем данные

In [ ]:
more = True
code = auth()
stop_time = int(time.time()) # API работает с unixtime
start_time = stop_time - 86400
vacancies = []
iteration = 0
while more == True:
    for p in range(12):
        print(f'Итерация №{iteration}, запрос №{p}') # логируем происходящее
        vac_list = get_vacancy(p, start_time, stop_time) # ходим в ручку
        print(f'Код ответа: {vac_list.status_code}')
        if vac_list.status_code != 200: # обработка ошибок
            code = auth()
            vac_list = get_vacancy(p, start_time, stop_time)
            if vac_list.status_code != 200:
                more = False
                print('Ошибка, выходим из цикла')
                break
        vac_list = vac_list.json()
        vacancies.extend(vac_list['objects'])
        print(f'Количество вакансий: {len(vacancies)}')
        more = vac_list['more'] # получаем инфу, есть ли ещё вакансии неполученные
        print(f'more = {more}')
        if more == False:
            break
        if p == 11:
            stop_time = vac_list['objects'][-1]['date_published'] # так как есть ограничение на 500 объектов по одному запросу (это как раз 12 страниц), двигаемся по дате
            start_time = stop_time - 86400
        time.sleep(0.5) # чтобы не словить блокировку IP
    iteration += 1
df = pd.DataFrame(vacancies)

In [ ]:
df.to_csv('dataset.csv')